# 01 - Data Ingestion and Exploratory Data Analysis (EDA)

**Purpose:**
- Load raw Medicare provider dataset (`Healthcare Providers.csv`)
- Validate schema and handle any missing values
- Extract the 7 standardized numerical features
- Generate exploratory data visualizations (Distribution and Correlation Heatmap) for the manuscript (Figures 1 and 2)

**Inputs:**
- `data/raw/healthcare_providers.csv`

**Outputs:**
- `data/processed/features_raw.parquet`
- `outputs/figures/fig1_distribution.png`
- `outputs/figures/fig2_correlation_heatmap.png`\n

In [ ]:
# Cell 01: Mount Storage & Bootstrap Paths
import os
import sys
from pathlib import Path

# Dual-Environment Parity Setup
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    # Assuming user works in a specific folder, e.g., Paper1_Revision
    BASE_DIR = Path('/content/drive/MyDrive/Paper1_Revision')
else:
    # Local execution
    BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Bootstrap expected directories
for folder in ['data/raw', 'data/interim', 'data/processed', 
               'outputs/figures', 'outputs/tables', 'outputs/models', 
               'outputs/notebook_exports']:
    (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)
    
print(f"Base directory set to: {BASE_DIR}")\n

In [ ]:
# Cell 02: Imports, Global Seeds & Style
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Global styling for manuscript-quality figures
plt.style.use('default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'figure.dpi': 300,        # High resolution for publication
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.autolayout': True # Prevents clipping of labels
})\n

In [ ]:
# Cell 03: Load Raw Data
raw_data_path = BASE_DIR / 'data' / 'raw' / 'healthcare_providers.csv'

print(f"Loading data from {raw_data_path}...")
df_raw = pd.read_csv(raw_data_path, low_memory=False)

print(f"Raw data shape: {df_raw.shape}")
assert df_raw.shape[0] == 100000, "Expected exactly 100,000 records"

df_raw.head()\n

In [ ]:
# Cell 04: Extract 7 Numerical Features
# These are the exact 7 features used in the original paper (MS 21575)
features = [
    'Number of Services',
    'Number of Medicare Beneficiaries',
    'Number of Distinct Medicare Beneficiary/Per Day Services',
    'Average Medicare Allowed Amount',
    'Average Submitted Charge Amount',
    'Average Medicare Payment Amount',
    'Average Medicare Standardized Amount'
]

df_features = df_raw[features].copy()

print(f"Features data shape: {df_features.shape}")
assert df_features.shape == (100000, 7), "Expected exactly (100,000, 7)"

# Check for missing values - Documentation Fix DOC-05
missing_counts = df_features.isna().sum()
print("\nMissing values per feature:")
print(missing_counts)
assert missing_counts.sum() == 0, "Expected zero missing values in these 7 features"

# Although there are no missing values, we add a precautionary fillna to match original intent
df_features = df_features.fillna(df_features.median())\n

In [ ]:
# Cell 05: Plot Feature Distributions (Figure 1)
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.histplot(df_features[col], bins=50, kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()

# Export figure
fig1_path = BASE_DIR / 'outputs' / 'figures' / 'fig1_distribution.png'
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
print(f"Saved Figure 1 to {fig1_path}")
plt.show()\n

In [ ]:
# Cell 06: Plot Correlation Heatmap (Figure 2)
plt.figure(figsize=(10, 8))
corr_matrix = df_features.corr(method='pearson')

# Mask the upper triangle for cleaner visualization
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, 
    mask=mask,
    annot=True, 
    fmt=".2f", 
    cmap='coolwarm', 
    vmin=-1, vmax=1, 
    square=True, 
    linewidths=.5,
    cbar_kws={"shrink": .8}
)
plt.title('Correlation Matrix of 7 Selected Features', pad=20)

# Export figure
fig2_path = BASE_DIR / 'outputs' / 'figures' / 'fig2_correlation_heatmap.png'
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
print(f"Saved Figure 2 to {fig2_path}")
plt.show()\n

In [ ]:
# Cell 07: Export Processed Features Matrix
# This parquet file acts as the deterministic input for 02_baseline_models.ipynb
export_path = BASE_DIR / 'data' / 'processed' / 'features_raw.parquet'
df_features.to_parquet(export_path, index=False)

print(f"Successfully exported {df_features.shape} matrix to {export_path}")\n